In [1]:
# import required libs
import numpy as np
import sys
sys.path.append(r"../Communiaction")
sys.path.append(r"../Utils")
sys.path.append(r"../../NeuralNetworks/Utils")
import TCP_IP_UTILS
import NN_Utils
import Utils
import time
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import struct
from torch.cuda.amp import autocast

In [2]:
#Add power supply code here -> DC Sources
import sys
import pyvisa
sys.path.append(r"../PowerSupply")
import PS_Utils
Device_ID_DC = "USB0::0x2A8D::0x1202::MY59003914::0::INSTR"
rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("Available VISA resources:")
for r in resources:
        print(" -", r)
device_dc = PS_Utils.connect_E36313A(Device_ID_DC)
DIG_LS = 1
ANA_LS = 2
MAIN = 3

Available VISA resources:
 - USB0::0x0957::0x1F01::MY53270560::0::INSTR
 - USB0::0x0957::0x1F01::MY57280340::0::INSTR
 - USB0::0x2A8D::0x1202::MY59003914::0::INSTR
 - USB0::0x2A8D::0x9007::MY62310119::0::INSTR
 - USB0::0x2A8D::0x9201::MY61391394::0::INSTR
Connected to: Keysight Technologies,E36313A,MY59003914,2.1.0-1.0.4-1.12


In [3]:
#connect to ethernet socket 
HOST = '192.168.1.10'  # Remote device IP (server IP)
PORT = 7               # Echo port
client_socket = TCP_IP_UTILS.tcp_connect(HOST,PORT)
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)
time.sleep(1) #Wait for IOs to settle

[Client] Connected to 192.168.1.10:7
Default Signals Set
[128]


In [4]:
#First Turn on Digital Leval Shifters
PS_Utils.set_voltage_E36313A(device_dc,DIG_LS,3,0.02)
print(PS_Utils.get_values_E36313A(device_dc,DIG_LS))
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

['+2.99775900E+00', '+5.47600000E-03']
Default Signals Set
[128]


In [5]:
#Turn on Main Chip PS
PS_Utils.set_voltage_E36313A(device_dc,MAIN,3,0.02)
print(PS_Utils.get_values_E36313A(device_dc,MAIN))

['+2.99735100E+00', '+1.27990000E-02']


In [6]:
#Turn on CLK 
#Device_ID_CLK = "USB0::0x0957::0x1F01::MY57280340::0::INSTR"
Device_ID_CLK = "USB0::0x0957::0x1F01::MY53270560::0::INSTR"
device_clk = PS_Utils.connect_N5173B(Device_ID_CLK)
PS_Utils.set_voltage_N5173B(device_clk,0.45)
PS_Utils.set_frequency_N5173B(device_clk,1000000000)
PS_Utils.enable_output_N5173B(device_clk)

Connected to: Agilent Technologies, N5173B, MY53270560, B.01.70


In [7]:
#Call DL EN Scan Chain here with data
scan_id = Utils.DL_EN_SC_LM["id"]
scan_len = Utils.DL_EN_SC_LM["len"]
scan_data = np.ones(scan_len,dtype=int).tolist()
#scan_data[4] = 0
#scan_data[2] = 0
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
print(ret_data)

[72, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received
[128]


In [8]:
#Call DL EN Scan Chain here with data
scan_id = Utils.DL_EN_SC_RM["id"]
scan_len = Utils.DL_EN_SC_RM["len"]
scan_data = np.ones(scan_len,dtype=int).tolist()
#scan_data[71] = 0
#scan_data[4] = 0
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
print(ret_data)

[72, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received
[128]


In [9]:
#Control Logic TD Data
import numpy as np
import random
import re
def scan_gen(time):
    SC_data = f"""
    IN_EXT_LOC = 1'b0;
    TDC_EN_EXT_LOC = 1'b0;
    ENCHG_EXT_LOC = 1'b0;
    ENTIME_EXT_LOC = 1'b1;
    PCHG_EXT_LOC = 1'b0;
    VDAC_CTRL_EXT_LOC = 1'b0;
    DEL_RST_EXT_LOC = 1'b0;
    TDC_COMPUTE_EXT_LOC = 1'b0;
    TDC_RST_EXT_LOC = 1'b0;
    VTC_EN_EXT_LOC = 1'b0;

    IN_LB = 6'd3;
    IN_UB = 6'd22; 
    TDC_EN_LB = 6'd1;
    TDC_EN_UB = 6'd21;
    ENCHG_LB = 6'd0;
    ENCHG_UB = 6'd0;
    ENTIME_LB = 6'd0;
    ENTIME_UB = 6'd0;
    PCHG_LB = 6'd2;
    PCHG_UB = 6'd22;
    VDAC_CTRL_LB = 6'd0;
    VDAC_CTRL_UB = 6'd0; 
    DEL_RST_LB = 6'd2;
    DEL_RST_UB = 6'd22;
    TDC_COMPUTE_LB = 6'd19;
    TDC_COMPUTE_UB = 6'd24;
    TDC_RST_LB = 6'd1;
    TDC_RST_UB = 6'd21;
    BL_NUM_TGT = 2'd3;
    BL_DONE_TIME = 6'd55;
    VTC_EN_LB = 6'd0;
    VTC_EN_UB = 6'd0;
    SAMPLE_EDGE_TIME = 6'd{time};

    CHG_TIME = 1'd0;
    OSC_DEL = 1'd0;
    """
    SC_order = ["BL_DONE_TIME","SAMPLE_EDGE_TIME", "BL_NUM_TGT", "BLANK_4", "IN_LB", "TDC_EN_LB", "ENCHG_LB", "IN_EXT_LOC","TDC_EN_EXT_LOC","ENCHG_EXT_LOC",\
                "ENTIME_EXT_LOC","PCHG_EXT_LOC","VDAC_CTRL_EXT_LOC", 'ENTIME_LB', 'PCHG_LB', 'VDAC_CTRL_LB', 'DEL_RST_LB', 'TDC_COMPUTE_LB', 'TDC_RST_LB', \
                'VTC_EN_LB',"DEL_RST_EXT_LOC","TDC_COMPUTE_EXT_LOC","TDC_RST_EXT_LOC","VTC_EN_EXT_LOC","CHG_TIME","OSC_DEL", "VTC_EN_UB","TDC_RST_UB","TDC_COMPUTE_UB",\
                "DEL_RST_UB","VDAC_CTRL_UB","PCHG_UB","ENTIME_UB","ENCHG_UB","TDC_EN_UB","IN_UB","BLANK_12_PISO"]

    lines = SC_data.splitlines()
    sc_signal = {}
    for line in lines:
        if line.strip():  # Skip empty lines
            # Handle 'd' (decimal) format
            match = re.match(r"(\w+) = (\d+)'d([0-9]+);", line.strip())
            if match:
                signal_name = match.group(1)
                num_bits = int(match.group(2))  # Number of bits
                signal_value = match.group(3)  # Decimal value


                # If there's only 1 bit, do not add the bit-range
                if num_bits == 1:
                    name = signal_name
                else:
                    # Format the name with the bit-range (e.g., VTC_EN_UB<[5:0]>)
                    name = "%s" % (signal_name) # Using 0-based index for the range

                # Format the decimal value to binary and pad to the correct number of bits
                binary_value = bin(int(signal_value))[2:].zfill(num_bits)
                sc_signal[name] = binary_value


            # Handle 'b' (binary) format
            elif "b" in line:  # e.g., BL_NUM_TGT = 2'b01;
                match = re.match(r"(\w+) = (\d+)'b([01]+);", line.strip())
                if match:
                    signal_name = match.group(1)
                    num_bits = int(match.group(2))  # Number of bits
                    signal_value = match.group(3)  # Binary value

                    # If there's only 1 bit, do not add the bit-range
                    if num_bits == 1:
                        name = signal_name
                    else:
                        # Format the name with the bit-range (e.g., BL_NUM_TGT<[1:0]> for 2 bits)
                        name = "%s" % (signal_name)

                    # Format the binary value to match the number of bits
                    binary_value = signal_value.zfill(num_bits)
                    sc_signal[name] = binary_value


    # print(sc_signal)
    output_list = []
    for signal in SC_order:
        if "BLANK" in signal:
            num_blanks = re.findall(r'\d+', signal)
            output_list += [0]*int(num_blanks[0])
        else:
            temp = sc_signal[signal][::-1]
            output_list += list(map(int,temp))
    print(sc_signal["SAMPLE_EDGE_TIME"])
    output_list = output_list[::-1] + [0]*30
    return output_list

def set_CS(cs):
    ret_data = 0
    if cs == 0:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [0,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 1:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [1,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 2:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [0,1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 3:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [1,1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    return ret_data

In [10]:
#Call Scan IN here with data
scan_id = Utils.CONTROL_LOGIC_SC["id"]
scan_len = Utils.CONTROL_LOGIC_SC["len"]
scan_data = scan_gen(5)
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
#print(ret_data)

000101
[162, 0]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Received


In [22]:
#Initialise SRAM with default data.
path = r"..\Utils\CalibrationData.xlsx"
write_data_default = Utils.get_Default_Write_Data(path).tolist()
ret_data = Utils.Write_SRAM(client_socket,write_data_default,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 6
Received Data Size: 1
All Data Received
[128]


In [12]:
#Turn on Analog PS
PS_Utils.set_voltage_E36313A(device_dc,ANA_LS,3,0.04)
print(PS_Utils.get_values_E36313A(device_dc,ANA_LS))

['+2.99815000E+00', '+2.47600000E-02']


In [13]:
flattened = (Utils.TD_CORRECTION).flatten()
ret_data = Utils.Send_CorrectionData(client_socket,flattened,0)
print(ret_data)

[128]


In [14]:
from torchvision import datasets, transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

train_dataset = datasets.CIFAR10(root=r"../../NeuralNetworks/Dataset/CIFAR10", train=True, download=False, transform=transform_train)
test_dataset = datasets.CIFAR10(root=r"../../NeuralNetworks/Dataset/CIFAR10", train=False, download=False, transform=transform_test)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class ResNet20(nn.Module):
    def __init__(self,use_custom = [],precision = [],HW_Modelled = [], client_socket = None, num_classes=10, LUT_FLAGS = [], LUT_PATHS = []):
        super(ResNet20, self).__init__()
        self.use_custom = use_custom
        self.precision = precision
        self.HW_Modelled = HW_Modelled
        self.client_socket = client_socket
        self.LUT_FLAGS = LUT_FLAGS
        self.LUT_PATHS = LUT_PATHS

        self.conv1 = nn.Conv2d(3, 16, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        # ---------- Stage 1 ----------
        self.conv2_1 = nn.Conv2d(16, 16, 3, 1, 1, bias=False)
        self.bn2_1 = nn.BatchNorm2d(16)
        self.conv2_2 = nn.Conv2d(16, 16, 3, 1, 1, bias=False)
        self.bn2_2 = nn.BatchNorm2d(16)

        self.conv3_1 = nn.Conv2d(16, 16, 3, 1, 1, bias=False)
        self.bn3_1 = nn.BatchNorm2d(16)
        self.conv3_2 = nn.Conv2d(16, 16, 3, 1, 1, bias=False)
        self.bn3_2 = nn.BatchNorm2d(16)

        self.conv4_1 = nn.Conv2d(16, 16, 3, 1, 1, bias=False)
        self.bn4_1 = nn.BatchNorm2d(16)
        self.conv4_2 = nn.Conv2d(16, 16, 3, 1, 1, bias=False)
        self.bn4_2 = nn.BatchNorm2d(16)

        # ---------- Stage 2 ----------
        self.conv5_1 = nn.Conv2d(16, 32, 3, 2, 1, bias=False)
        self.bn5_1 = nn.BatchNorm2d(32)
        self.conv5_2 = nn.Conv2d(32, 32, 3, 1, 1, bias=False)
        self.bn5_2 = nn.BatchNorm2d(32)
        self.shortcut5 = nn.Conv2d(16, 32, 1, 2, 0, bias=False)
        self.shortcut_bn5 = nn.BatchNorm2d(32)

        self.conv6_1 = nn.Conv2d(32, 32, 3, 1, 1, bias=False)
        self.bn6_1 = nn.BatchNorm2d(32)
        self.conv6_2 = nn.Conv2d(32, 32, 3, 1, 1, bias=False)
        self.bn6_2 = nn.BatchNorm2d(32)

        self.conv7_1 = nn.Conv2d(32, 32, 3, 1, 1, bias=False)
        self.bn7_1 = nn.BatchNorm2d(32)
        self.conv7_2 = nn.Conv2d(32, 32, 3, 1, 1, bias=False)
        self.bn7_2 = nn.BatchNorm2d(32)

        # ---------- Stage 3 ----------
        self.conv8_1 = nn.Conv2d(32, 64, 3, 2, 1, bias=False)
        self.bn8_1 = nn.BatchNorm2d(64)
        self.conv8_2 = nn.Conv2d(64, 64, 3, 1, 1, bias=False)
        self.bn8_2 = nn.BatchNorm2d(64)
        self.shortcut8 = nn.Conv2d(32, 64, 1, 2, 0, bias=False)
        self.shortcut_bn8 = nn.BatchNorm2d(64)

        self.conv9_1 = nn.Conv2d(64, 64, 3, 1, 1, bias=False)
        self.bn9_1 = nn.BatchNorm2d(64)
        self.conv9_2 = nn.Conv2d(64, 64, 3, 1, 1, bias=False)
        self.bn9_2 = nn.BatchNorm2d(64)

        self.conv10_1 = nn.Conv2d(64, 64, 3, 1, 1, bias=False)
        self.bn10_1 = nn.BatchNorm2d(64)
        self.conv10_2 = nn.Conv2d(64, 64, 3, 1, 1, bias=False)
        self.bn10_2 = nn.BatchNorm2d(64)

        # ---------- Final ----------
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        # Initial Conv
        x = F.relu(self.bn1(self.conv1(x)))
        sum1 = None
        # Stage 1
        identity = x
        if self.use_custom[0]:
            custom_conv0 = NN_Utils.Convolution(self.HW_Modelled[0],x,self.conv2_1.weight,self.precision[0],self.conv2_1.stride[0],self.conv2_1.padding[0],self.conv2_1.bias,self.client_socket,self.LUT_FLAGS[0],self.LUT_PATHS[0])
            x,sum1 = custom_conv0.convolution_onChip()
        else:
            x = self.conv2_1(x)
        out = F.relu(self.bn2_1(x))
        if self.use_custom[1]:
            custom_conv1 = NN_Utils.Convolution(self.HW_Modelled[1],out,self.conv2_2.weight,self.precision[1],self.conv2_2.stride[0],self.conv2_2.padding[0],self.conv2_2.bias,self.client_socket,self.LUT_FLAGS[1],self.LUT_PATHS[1])
            out,sum1 = custom_conv1.convolution_onChip()
        else:
            out = self.conv2_2(out)
        out = self.bn2_2(out)
        x = F.relu(out + identity)

        identity = x
        if self.use_custom[2]:
            custom_conv2 = NN_Utils.Convolution(self.HW_Modelled[2],x,self.conv3_1.weight,self.precision[2],self.conv3_1.stride[0],self.conv3_1.padding[0],self.conv3_1.bias,self.client_socket,self.LUT_FLAGS[2],self.LUT_PATHS[2])
            x,sum1 = custom_conv2.convolution_onChip()
        else:
            x = self.conv3_1(x)
        out = F.relu(self.bn3_1(x))
        if self.use_custom[3]:
            custom_conv3 = NN_Utils.Convolution(self.HW_Modelled[3],out,self.conv3_2.weight,self.precision[3],self.conv3_2.stride[0],self.conv3_2.padding[0],self.conv3_2.bias,self.client_socket,self.LUT_FLAGS[3],self.LUT_PATHS[3])
            out,sum1 = custom_conv3.convolution_onChip()
        else:
            out = self.conv3_2(out)
        out = self.bn3_2(out)
        x = F.relu(out + identity)

        identity = x
        if self.use_custom[4]:
            custom_conv4 = NN_Utils.Convolution(self.HW_Modelled[4],x,self.conv4_1.weight,self.precision[4],self.conv4_1.stride[0],self.conv4_1.padding[0],self.conv4_1.bias,self.client_socket,self.LUT_FLAGS[4],self.LUT_PATHS[4])
            x,sum1 = custom_conv4.convolution_onChip()
        else:
            x = self.conv4_1(x)
        out = F.relu(self.bn4_1(x))
        if self.use_custom[5]:
            custom_conv5 = NN_Utils.Convolution(self.HW_Modelled[5],out,self.conv4_2.weight,self.precision[5],self.conv4_2.stride[0],self.conv4_2.padding[0],self.conv4_2.bias,self.client_socket,self.LUT_FLAGS[5],self.LUT_PATHS[5])
            out,sum1 = custom_conv5.convolution_onChip()
        else:
            out = self.conv4_2(out)
        out = self.bn4_2(out)
        x = F.relu(out + identity)

        # Stage 2
        identity = self.shortcut_bn5(self.shortcut5(x))
        out = F.relu(self.bn5_1(self.conv5_1(x)))
        out = self.bn5_2(self.conv5_2(out))
        x = F.relu(out + identity)

        identity = x 
        if self.use_custom[6]:
            custom_conv6 = NN_Utils.Convolution(self.HW_Modelled[6],x,self.conv6_1.weight,self.precision[6],self.conv6_1.stride[0],self.conv6_1.padding[0],self.conv6_1.bias,self.client_socket,self.LUT_FLAGS[6],self.LUT_PATHS[6])
            x,sum1 = custom_conv6.convolution_onChip()
        else:
            x = self.conv6_1(x)
        out = F.relu(self.bn6_1(x))
        if self.use_custom[7]:
            custom_conv7 = NN_Utils.Convolution(self.HW_Modelled[7],out,self.conv6_2.weight,self.precision[7],self.conv6_2.stride[0],self.conv6_2.padding[0],self.conv6_2.bias,self.client_socket,self.LUT_FLAGS[7],self.LUT_PATHS[7])
            out,sum1 = custom_conv7.convolution_onChip()
        else:
            out = self.conv6_2(out)
        out = self.bn6_2(out)
        x = F.relu(out + identity)

        identity = x
        if self.use_custom[8]:
            custom_conv8 = NN_Utils.Convolution(self.HW_Modelled[8],x,self.conv7_1.weight,self.precision[8],self.conv7_1.stride[0],self.conv7_1.padding[0],self.conv7_1.bias,self.client_socket,self.LUT_FLAGS[8],self.LUT_PATHS[8])
            x,sum1 = custom_conv8.convolution_onChip()
        else:
            x = self.conv7_1(x)
        out = F.relu(self.bn7_1(x))
        if self.use_custom[9]:
            custom_conv9 = NN_Utils.Convolution(self.HW_Modelled[9],out,self.conv7_2.weight,self.precision[9],self.conv7_2.stride[0],self.conv7_2.padding[0],self.conv7_2.bias,self.client_socket,self.LUT_FLAGS[9],self.LUT_PATHS[9])
            out,sum1 = custom_conv9.convolution_onChip()
        else:
            out = self.conv7_2(out)
        out = self.bn7_2(out)
        x = F.relu(out + identity)

        # Stage 3
        identity = self.shortcut_bn8(self.shortcut8(x))
        out = F.relu(self.bn8_1(self.conv8_1(x)))
        out = self.bn8_2(self.conv8_2(out))
        x = F.relu(out + identity)

        identity = x
        if self.use_custom[10]:
            custom_conv10 = NN_Utils.Convolution(self.HW_Modelled[10],x,self.conv9_1.weight,self.precision[10],self.conv9_1.stride[0],self.conv9_1.padding[0],self.conv9_1.bias,self.client_socket,self.LUT_FLAGS[10],self.LUT_PATHS[10])
            x,sum1 = custom_conv10.convolution_onChip()
        else:
            x = self.conv9_1(x)
        out = F.relu(self.bn9_1(x))
        if self.use_custom[11]:
            custom_conv11 = NN_Utils.Convolution(self.HW_Modelled[11],out,self.conv9_2.weight,self.precision[11],self.conv9_2.stride[0],self.conv9_2.padding[0],self.conv9_2.bias,self.client_socket,self.LUT_FLAGS[11],self.LUT_PATHS[11])
            out,sum1 = custom_conv11.convolution_onChip()
        else:
            out = self.conv9_2(out)
        out = self.bn9_2(out)
        x = F.relu(out + identity)

        identity = x
        if self.use_custom[12]:
            custom_conv12 = NN_Utils.Convolution(self.HW_Modelled[12],x,self.conv10_1.weight,self.precision[12],self.conv10_1.stride[0],self.conv10_1.padding[0],self.conv10_1.bias,self.client_socket,self.LUT_FLAGS[12],self.LUT_PATHS[12])
            x,sum1 = custom_conv12.convolution_onChip()
        else:
            x = self.conv10_1(x)
        out = F.relu(self.bn10_1(x))
        if self.use_custom[13]:
            custom_conv13 = NN_Utils.Convolution(self.HW_Modelled[13],out,self.conv10_2.weight,self.precision[13],self.conv10_2.stride[0],self.conv10_2.padding[0],self.conv10_2.bias,self.client_socket,self.LUT_FLAGS[13],self.LUT_PATHS[13])
            out,sum1 = custom_conv13.convolution_onChip()
        else:
            out = self.conv10_2(out)
        out = self.bn10_2(out)
        x = F.relu(out + identity)

        # Final pooling and FC
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x,sum1

In [16]:

TD_DEL_LUT_PATH = r"../../NeuralNetworks/Utils/Hardware_Model/HW_MODEL_TD_DEL.npy"
TD_LUT_PATH = r"../../NeuralNetworks/Utils/Hardware_Model/HW_MODEL_TD_FastIMC.npy" #r"../Utils/Hardware_Model/HW_MODEL_TD_1x_Ref256.npy"
CD_LUT_PATH = r"../../NeuralNetworks/Utils/Hardware_Model/HW_MODEL_CD_FastIMC.npy" #r"../Utils/Hardware_Model/HW_MODEL_CD_4x_Ref256_sweep4.npy"
LUT_PATHS = [[TD_DEL_LUT_PATH,CD_LUT_PATH]]*14
LUT_FLAGS = [[False,False,False]]*14
#LUT_FLAGS[9] = [False,True,True]
#LUT_PATHS[9] = [TD_LUT_PATH,CD_LUT_PATH]

In [17]:
correct = 0
total = 0
count = 0

In [18]:
#torch.backends.cudnn.deterministic = True
#torch.backends.cudnn.benchmark = False
HW_Modelled = [1,1,1,1,1,1,1,1,1,1,1,1,1,0]
precision = [[0,0]]*14
use_custom=[False]*14
use_custom[13] = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet20(HW_Modelled=HW_Modelled,use_custom=use_custom,precision=precision,num_classes=10,client_socket=client_socket,LUT_FLAGS=LUT_FLAGS,LUT_PATHS=LUT_PATHS).to(device)
model.load_state_dict(torch.load( r"../../NeuralNetworks/Model_Zoo/Resnet20/resnet20_cifar10.pth", map_location=device))
# 🧩 Step 8: Evaluate on Test Set
model.eval()
torch.cuda.empty_cache()
correct = 0
total = 0
count = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs,sum1 = model(imgs)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        count += 1
        print(f"Batches Done : {count} Correct : {correct} Total : {total} Inference Accuracy: {100 * correct / total:.2f}%\n\n")
        if count == 1: break
torch.cuda.empty_cache()
print(f"Batches Done : {count} Correct : {correct} Total : {total} Final Inference Accuracy: {100 * correct / total:.2f}%\n\n")
del model

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 8
Received Data Size: 16384
All Data Received
Batches Done : 1 Correct : 1 Total : 1 Inference Accuracy: 100.00%


Batches Done : 1 Correct : 1 Total : 1 Final Inference Accuracy: 100.00%




In [24]:
#Call Read SRAM here
SRAM_DATA = Utils.Read_SRAM(client_socket,1)
print(len(SRAM_DATA))
print(SRAM_DATA)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 5
Received Data Size: 46240
All Data Received
46240
[85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 85, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [25]:
unique = set(SRAM_DATA)
print(unique)

{0, 85}


In [21]:
#Set IO to default and exit    
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

Default Signals Set
[128]


In [23]:
#Initialise SRAM with default data.
path = r"..\Utils\CalibrationData.xlsx"
write_data_default = Utils.get_Default_Write_Data(path).tolist()
ret_data = Utils.Write_SRAM(client_socket,write_data_default,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 6
Received Data Size: 1
All Data Received
[128]


In [26]:
PS_Utils.kill_channel_E36313A(device_dc,ANA_LS)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,MAIN)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,DIG_LS)
PS_Utils.disconnect_E36313A(device_dc)
time.sleep(0.3)
PS_Utils.kill_N5173B(device_clk)
PS_Utils.disconnect_N5173B(device_clk)

0